In [ ]:
import warnings
import os

# Suppress ArviZ and PyMC noisy warnings
warnings.filterwarnings("ignore")

# Suppress OpenMP duplicate library warning
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# full diagnostic workflow
# def model -> prior predictive check -> sample -> convergence audit -> posterior predictive check

TRUE_MU, TRUE_SIGMA = 30.0, 4.0
cube_tests = np.random.normal(TRUE_MU, TRUE_SIGMA, 50)

# 1. Define Model and Sample
with pm.Model() as full_workflow_model:
    mu = pm.Normal("mu", mu=30, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=10)
    y = pm.Normal("y", mu=mu, sigma=sigma, observed=cube_tests)

# 2.Pior checks
    # Sample from the priors ONLY (no data influence)
    prior_checks = pm.sample_prior_predictive(samples=1000, random_seed=42)
    #  "Are my initial assumptions physically sane?"
    # that is do the data gen from my assumption pass tbhe sanity check

# always use bubble graphs; #display(pm.model_to_graphviz(model_name)) to visualize dag and taht arrows make sense

# Use ArviZ for high-signal visualizations.
# az.plot_ppc overlays the observed data against the prior predictive distributions.
fig, ax = plt.subplots(figsize=(12, 6))
az.plot_ppc(prior_checks,
group="prior",
kind="kde", # kde = kernel density estimate
ax=ax,
colors=['gray', 'black', 'blue'], alpha=0.8) # [Prior, Mean, Observed]

az.plot_kde(cube_tests, ax=ax, plot_kwargs={"color": "red", "linewidth": 3, "linestyle": "--"}, label="Actual Observed Data")

ax.set_title("Prior Predictive Check: Physicality Audit", fontsize=14, fontweight='bold')
ax.set_xlabel("Concrete Strength (MPa)", fontsize=12)
ax.set_ylabel("Probability Density", fontsize=12)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

# we want our priors to be wider/diffuse than observed
# if priors are too tight, the model will refuse to listen even if
# real data is slightliy outside the tight outside bounds; 'overconfidence bias'

# litil bleeding past 0, better use other distro, but for now nromal ok

In [ ]:
# using hist for prior checks
# but kde on prior better cuz we dealing with massive uncertainity
# and goal is to show envelope of possibility, x domain

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
prior_y = prior_checks.prior_predictive["y"].values.flatten()
axes[0].hist(prior_y, bins=50, density=True, alpha=0.6, color='gray')
axes[0].axvline(cube_tests.mean(), color='r', linestyle='--', label='Actual data mean')
axes[0].set_title("Prior Predictive: What data do my priors expect?")
axes[0].set_xlabel("Concrete Strength (MPa)")
axes[0].legend()

# Plot the actual data for comparison
axes[1].hist(cube_tests, bins=20, density=True, alpha=0.6, color='blue')
axes[1].set_title("Actual Observed Data")
axes[1].set_xlabel("Concrete Strength (MPa)")

plt.tight_layout()
plt.show()

In [ ]:
# 3 sample
with full_workflow_model:
    trace = pm.sample(2000, tune=1000, random_seed=42, progressbar=False, chains=4, cores=4) # industry std is 4


In [ ]:
# 4 convergence/divergence


# chains = survyers, cores = helicopters

az.plot_trace(trace, var_names=["mu", "sigma"])

#  **GOOD trace:** "Fuzzy caterpillar" — chain bounces randomly. Distribution is smooth. (we got this)
# **BAD trace:** Chain drifts, trends, or gets stuck. Distribution is jagged.

plt.suptitle("Convergence Check: Trace Plots")
plt.tight_layout()
plt.show()



print(az.summary(trace, var_names=["mu", "sigma"]))

# r_hat ≈ 1.00 = GOOD.  r_hat > 1.05 = BAD (chains disagree)

# r hat is ratio of total var (within chains+inter chain) / within chain

# ess_bulk > 400 = GOOD.  ess_bulk < 100 = BAD (too few effective samples)

divergences = trace.sample_stats["diverging"].values.sum()
# even one divergence in 8k sample INVALIDATES a fidic risk model
print(f"\nCRITICAL AUDIT - Total Divergences: {divergences}")
if divergences > 0:
    print("WARNING: Model geometry is fractured. Refactor DAG or increase target_accept.")
else:
    print("STATUS: Geometry is sound. Cleared for Posterior Predictive.")

In [ ]:
# 5 post predictive check

with full_workflow_model:
    # Generate FAKE data using the fitted posterior parameters
    posterior_checks = pm.sample_posterior_predictive(trace, random_seed=42, progressbar=False)

# Visualize: Does data generated by the model look like the real data?
az.plot_ppc(posterior_checks, observed_rug=True, kind="cumulative")

# here we use cumulative s curve to avoid kde smoothing at edges, isntant lookup of tail percentiles
# and easy visual check of exact reporduciton of dataset
# and probab mapped in y axis

plt.title("Posterior Predictive Check: Model vs Reality")
plt.tight_layout()
plt.show()
# ecdf = empricially distro cumulative fxn

# posterior predictive must completely bracket the observed data
# if NOT, bad likelihood choice, missing covariates, etc.

# the small vertical lines directly represents our actual observed data points
# its helpful for finding outliers, response of s curve to them, and see if the blue liens are expanding to cover outliers if present

In [ ]:

# flatten the posterior predictive samples into a single 1D array
simulated_reality = posterior_checks.posterior_predictive["y"].values.flatten()

#  Commercial Threshold Query (e.g., FIDIC limit is 25 MPa)
threshold = 25.0
p_fail = (simulated_reality < threshold).mean() * 100

print(f"Risk Audit: Probability of concrete testing below {threshold} MPa is {p_fail:.2f}%")

# 2. The Sovereign Percentile Query (e.g., What is our 95% worst-case minimum strength?)
# In numpy, percentile uses 0-100 scale. So 5th percentile is the bottom 5%.
p05_worst_case = np.percentile(simulated_reality, 5)

print(f"Contract Baseline: We are 95% confident the concrete will be at least {p05_worst_case:.2f} MPa.")

In [ ]:
# sabotage model with less tuning

with pm.Model() as broken_model:
    mu = pm.Normal("mu", mu=30, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=10)
    y = pm.Normal("y", mu=mu, sigma=sigma, observed=cube_tests)
    trace_broken = pm.sample(2000, tune=10, random_seed=42, progressbar=False)

# only 10 practice steps, and cant 'tune' its step size/scale to perfrom the draws

az.plot_trace(trace_broken)
az.summary(trace_broken)
# rhat 1.01 for the mu, ess bulk signifcantly decrease

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
az.plot_trace(trace_broken, var_names=["mu"], axes=axes[0:1])
az.plot_trace(trace, var_names=["mu"], axes=axes[1:2])
axes[0][0].set_title("BROKEN (tune=10)")
axes[1][0].set_title("FIXED (tune=1000)")
plt.tight_layout()
plt.show()

# we can see denser traces, and more alinged distribution

In [ ]:
# divergenet/conflicting model

with pm.Model() as divergent_model:
    mu = pm.Normal("mu", mu=0, sigma=0.001)  # VERY tight prior at 0
    # even at this NUTS is an apex pred algo, stil converge with r hat at 1, lietrally NUTS
    # takes a lot to break
    # even if widest of priors it works, must not be wrong
    sigma = pm.HalfNormal("sigma", sigma=10)
    y = pm.Normal("y", mu=mu, sigma=sigma, observed=cube_tests)
    trace_div = pm.sample(2000, tune=1000, cores=1, random_seed=42, chains=4, progressbar=False)

divergences_div = trace_div.sample_stats["diverging"].values.sum()
print(f"Number of divergences: {divergences_div}")

az.plot_trace(trace_div)
plt.subplots_adjust(hspace=0.4, wspace=0.3)
plt.show()
az.summary(trace_div)
